# 00 | Load FIRMS fire data

Loads MODIS and VIIRS hotspots into one table with the same columns and draws the first monthly counts.
The loading code lives in `syntra/data.py`; fix things there, not here.

Run all cells. To change country or years, edit the settings cell.

## 1. Settings

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # lets the notebook import syntra/

from syntra.data import load_fires, PROCESSED_DIR

COUNTRY = "Saudi_Arabia"   # as FIRMS writes it, spaces become underscores
START, END = 2000, 2025    # yearly files end at 2025; 2026 needs the API + MAP_KEY

## 2. One table for both sensors

Downloads any missing yearly files into `data/raw`, then merges them. MODIS confidence (0 to 100) is mapped to the VIIRS classes with the MODIS user guide thresholds (Table 11): below 30 low, 30 to 79 nominal, 80 and up high. Only type 0 (vegetation) is kept.

In [ ]:
fires = load_fires(COUNTRY, START, END)
print(f"{len(fires):,} vegetation hotspots")
fires.groupby("sensor").size()

## 3. Monthly counts per sensor (the gap we need to harmonize)

In [ ]:
import pandas as pd
import plotly.express as px

monthly = (fires.groupby(["sensor", pd.Grouper(key="acq_date", freq="MS")])
                .size().rename("hotspots").reset_index())
px.line(monthly, x="acq_date", y="hotspots", color="sensor",
        title=f"Monthly hotspot counts, {COUNTRY.replace('_', ' ')}",
        labels={"acq_date": "Month", "hotspots": "Hotspots per month"}).show()

overlap = monthly[monthly["acq_date"].dt.year >= 2012].pivot(index="acq_date", columns="sensor", values="hotspots")
print("Average VIIRS-to-MODIS ratio per month, overlap years (raw):")
print(overlap.drop(columns="MODIS").div(overlap["MODIS"], axis=0).replace(float("inf"), pd.NA).mean().round(2))

## 4. Fire calendar (MODIS only, raw)

In [ ]:
cal = (fires[fires["sensor"] == "MODIS"].groupby(["year", "month"]).size()
            .unstack(fill_value=0).reindex(columns=range(1, 13), fill_value=0))
cal.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
px.imshow(cal, color_continuous_scale="cividis", aspect="auto",
          labels={"x": "Month", "y": "Year", "color": "Hotspots"},
          title="MODIS hotspots per month (raw)").show()

## 5. Save the merged table to data/processed

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out = PROCESSED_DIR / f"firms_merged_{COUNTRY}.csv"
fires.to_csv(out, index=False)
print("Saved", out, f"({len(fires):,} rows)")

## Next

Phase 4: `01_harmonization.ipynb`, compare MODIS and VIIRS over 2012 to 2025 and build the conversion. Write what you find in the Claude Doc too.